# Cryptocurrency Portfolio Optimisation: A GARCH-EVT Forecasted CVaR constraint and Safe Reinforcement Learning Approach

This is the main jupyter notebook for my Bachelor Degree Dissertaion in University of Southampton. This note book is relied on the [FinRL](https://github.com/AI4Finance-Foundation/FinRL), which is also modified by me to better suit my research.

## Table of Contents

* [1. Problem Definition](#1)
* [2. Getting Started - Load Python packages](#2)
    * [2.1. Install Packages](#2.1)
    * [2.2. Import Packages](#2.2)
    * [2.3. Create Folders](#2.3)
* [3. Download Data](#3)
* [4. Preprocess Data]      
    * [4.1. Technical Indicators]
    * [4.2. Perform Feature Engineering]
* [5.Build Environment] 
    * [5.1. Training & Trade Data Split]
    * [5.2. User-defined Environment]
    * [5.3. Initialize Environment]
* [6.Implement DRL Algorithms]
* [7.Backtesting Performance]
    * [7.1. BackTestStats]
    * [7.2. BackTestPlot]
    * [7.3. Baseline Stats]
    * [7.3. Compare to Stock Market Index]

<a id='1'></a>
# Part 1. Problem Definition

This study leverages **GARCH(1,1) to predict future volatility** and computes the **forecasted Conditional Value at Risk (Forecasted CVaR, $ \widehat{\text{CVaR}} $)**, which is then integrated into a safe reinforcement learning (RL) framework for portfolio optimization. The entire problem is formulated as a **Markov Decision Process (MDP)** as follows:

$$
\mathcal{M} = (\mathcal{S}, \mathcal{A}, \mathcal{P}, \mathcal{R}, \gamma)
$$

where:
- **$ \mathcal{S} $** is the state space.
- **$ \mathcal{A} $** is the action space.
- **$ \mathcal{P} $** is the state transition function.
- **$ \mathcal{R} $** is the reward function.
- **$ \gamma $** is the discount factor.

**1. State Space ($ \mathcal{S} $)**

At time $ t $, the portfolio state is defined as:

$$
s_t = \{ P_t, O_t, H_t, L_t, C_t, V_t, W_t, I_t, M_t, \widehat{\text{CVaR}}_t \}
$$

where:
- **Price Information**:
  - $ O_t = (o_t^1, o_t^2, ..., o_t^n) $: Opening prices of all assets.
  - $ H_t = (h_t^1, h_t^2, ..., h_t^n) $: High prices of all assets.
  - $ L_t = (l_t^1, l_t^2, ..., l_t^n) $: Low prices of all assets.
  - $ C_t = (c_t^1, c_t^2, ..., c_t^n) $: Closing prices of all assets.
  - $ V_t = (v_t^1, v_t^2, ..., v_t^n) $: Volume of all assets.
- **Current Portfolio Holdings**:
  - $ W_t = (w_t^1, w_t^2, ..., w_t^n) $: Current asset allocations, satisfying $ \sum_{i=1}^{n} w_t^i = 1 $.
- **Market Trading Status**:
  - $ I_t = (i_t^1, i_t^2, ..., i_t^n) $: Indicator variables denoting whether an asset is tradable (e.g., stocks are not tradable on weekends).
- **Market Technical Indicators**:
  - $ M_t $: Includes **MACD**, **RSI**, **moving averages (MA)**, etc.
- **Forecasted CVaR**:
  - Future **CVaR** calculated using **GARCH-predicted volatility**:
    $$
    \widehat{\text{CVaR}}_t = -\left(\Phi^{-1}(\alpha) + \frac{\phi(\Phi^{-1}(\alpha))}{1 - \alpha} \right) \cdot \widehat{\sigma}_{t+1}
    $$

**2. Action Space ($ \mathcal{A} $)**

Portfolio adjustments are made by modifying asset weights $ W_t $:

$$
a_t = (a_t^1, a_t^2, ..., a_t^n) \quad \text{where } \sum_{i=1}^{n} a_t^i = 1
$$

However, actions must pass through a **safety layer (Safety Layer)**, ensuring:
1. **Weight Bound Constraints**:
   $$
   0 \leq w_t^i \leq 1, \quad \forall i
   $$
2. **Cardinality Constraint**:
   $$
   \sum_{i=1}^{n} \mathbf{1}(w_t^i > 0) \leq K
   $$
   where $ K $ is the maximum number of selected assets.
3. **Non-Tradable Days Constraint**:
   $$
   a_t^i = w_t^i, \quad \text{if } I_t^i = 0
   $$
   ensuring that non-crypto assets cannot be adjusted on non-trading days.

The **safety layer** then applies a projection operator $ \Pi $:

$$
\tilde{a}_t = \Pi(a_t)
$$

**3. State Transition ($ \mathcal{P} $)**

Portfolio value $ P_t $ evolves based on asset price movements:

$$
P_{t+1} = P_t \sum_{i=1}^{n} w_t^i \frac{C_{t+1}^i}{C_t^i}
$$

If an asset is not tradable on a particular day, its price remains unchanged:

$$
C_{t+1}^i = C_t^i, \quad \text{if } I_t^i = 0
$$

GARCH-predicted future volatility follows the transition equation:

$$
\widehat{\sigma}_{t+1}^2 = \omega + \alpha \cdot r_t^2 + \beta \cdot \sigma_t^2
$$

**4. Reward Function ($ \mathcal{R} $)**

The reward function considers **portfolio returns, forecasted CVaR risk, and transaction costs**:

$$
R_t = P_{t+1} - \lambda \cdot \widehat{\text{CVaR}}_t - \delta_t
$$

where:
- **Portfolio Return**:
  $$
  P_{t+1} = P_t \sum_{i=1}^{n} w_t^i \frac{C_{t+1}^i}{C_t^i}
  $$
- **GARCH-Predicted Future CVaR**:
  $$
  \widehat{\text{CVaR}}_{\alpha} = -\left(\Phi^{-1}(\alpha) + \frac{\phi(\Phi^{-1}(\alpha))}{1 - \alpha} \right) \cdot \widehat{\sigma}_{t+1}
  $$
- **Transaction Costs**:
  $$
  \delta_t = \sum_{i=1}^{n} |w_t^i - w_{t-1}^i| \cdot \beta_i \cdot P_t
  $$
  where $ \beta_i $ represents the transaction cost coefficient for each asset.

**5. Discount Factor ($ \gamma $)**

To account for long-term returns, we introduce a discount factor $ \gamma \in (0,1) $:

$$
G_t = \sum_{k=0}^{\infty} \gamma^k R_{t+k}
$$

**Summary**

This study leverages **GARCH to predict future volatility and compute Forecasted CVaR**, which is integrated into a reinforcement learning-based portfolio optimization framework:
1. **Uses Forecasted CVaR to optimize risk-adjusted returns**, overcoming limitations of static CVaR.
2. **Incorporates a safety layer** to ensure realistic trading constraints.
3. **Considers market trading rules (e.g., stocks are non-tradable on weekends)** for enhanced real-world applicability.


<a id='2'></a>
# Part 2. Getting Started- Load Python Packages

<a id='2.1'></a>
## 2.1. Install Packages

See in requirement

<a id='2.2'></a>
## 2.2. Import Packages

In [49]:
import pandas as pd
import numpy as np
import datetime
import yfinance as yf
import os
from argparse import ArgumentParser
from typing import List
import copy
import datetime
from copy import deepcopy
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyfolio
from pyfolio import timeseries
!python ResetFolder.py

<a id='2.3'></a>
## 2.3. Create Folders

In [ ]:
import config

if not os.path.exists("./" + config.DATA_SAVE_DIR):
    os.makedirs("./" + config.DATA_SAVE_DIR)
if not os.path.exists("./" + config.TRAINED_MODEL_DIR):
    os.makedirs("./" + config.TRAINED_MODEL_DIR)
if not os.path.exists("./" + config.TENSORBOARD_LOG_DIR):
    os.makedirs("./" + config.TENSORBOARD_LOG_DIR)
if not os.path.exists("./" + config.RESULTS_DIR):
    os.makedirs("./" + config.RESULTS_DIR)

<a id='3'></a>
# Part 3. Download Data

In [ ]:
from config_tickers import CRYPTO_TICKER, DOW_30_TICKER
from config import TRAIN_START_DATE, TRAIN_END_DATE, TEST_END_DATE, TEST_START_DATE

def get_data(ticker_list: list, start_date: str, end_date: str) -> pd.DataFrame:
    for ticker in ticker_list:
        print(f"Downloading {ticker} data")
        data = yf.download(ticker, start=start_date, end=end_date)
        data.columns = data.columns.droplevel(1)
        data.rename(columns={'Date': 'Date'}, inplace=True)
        data.to_csv(f"{config.DATA_SAVE_DIR}/general/{ticker}.csv")

# split data into train and test and trade
def split_data(ticker_list: list) -> pd.DataFrame:
    for ticker in ticker_list:
        print(f"Splitting {ticker} data")
        data = pd.read_csv(f"{config.DATA_SAVE_DIR}/General/{ticker}.csv", index_col=0, parse_dates=True)
        train_data = data[(data.index >= TRAIN_START_DATE) & (data.index <= TRAIN_END_DATE)]
        train_data.to_csv(f"{config.DATA_SAVE_DIR}/train/{ticker}.csv")
        test_data = data[(data.index >= TEST_START_DATE) & (data.index <= TEST_END_DATE)]
        test_data.to_csv(f"{config.DATA_SAVE_DIR}/test/{ticker}.csv")

# Download Data begin
print("Downloading Data")
get_data(CRYPTO_TICKER, TRAIN_START_DATE, TEST_END_DATE)
get_data(DOW_30_TICKER, TRAIN_START_DATE, TEST_END_DATE)
# Split Data begin
print("Splitting Data")
split_data(CRYPTO_TICKER)
split_data(DOW_30_TICKER)

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


[*********************100%***********************]  1 of 1 completed


[*********************100%***********************]  1 of 1 completed


[*********************100%***********************]  1 of 1 completed


[*********************100%***********************]  1 of 1 completed


...

[*********************100%***********************]  1 of 1 completed


[*********************100%***********************]  1 of 1 completed


[*********************100%***********************]  1 of 1 completed


[*********************100%***********************]  1 of 1 completed


Splitting Data
Splitting BTC-USD data
Splitting ETH-USD data
Splitting XRP-USD data
Splitting USDT-USD data
Splitting SOL-USD data
Splitting BNB-USD data
Splitting USDC-USD data
Splitting DOGE-USD data
Splitting ADA-USD data
Splitting TRX-USD data
Splitting XLM-USD data
Splitting SHIB-USD data
Splitting AMZN data
Splitting AXP data
Splitting AMGN data
Splitting AAPL data
Splitting BA data
Splitting CAT data
Splitting CSCO data
Splitting CVX data
Splitting GS data
Splitting HD data
Splitting HON data
Splitting IBM data
Splitting JNJ data
Splitting KO data
Splitting JPM data
Splitting MCD data
Splitting MMM data
Splitting MRK data
Splitting MSFT data
Splitting NKE data
Splitting PG data
Splitting SHW data
Splitting TRV data
Splitting UNH data
Splitting CRM data
Splitting NVDA data
Splitting VZ data
Splitting V data
Splitting WMT data
Splitting DIS data


<a id='4'></a>
# Part 4. Preprocess Data

Data preprocessing is a crucial step for training a high quality machine learning model. We need to check for missing data and do feature engineering in order to convert the data into a model-ready state.
* Add technical indicators. In practical trading, various information needs to be taken into account, for example the historical stock prices, current holding shares, technical indicators, etc. In this article, we demonstrate two trend-following technical indicators: MACD and RSI.
* Add turbulence index. Risk-aversion reflects whether an investor will choose to preserve the capital. It also influences one's trading strategy when facing different market volatility level. To control the risk in a worst-case scenario, such as financial crisis of 2007–2008, FinRL employs the financial turbulence index that measures extreme asset price fluctuation.

<a id='4.1'></a>
## 4.1. Technical Indicators

<a id='4.2'></a>
## 4.2. Perform Feature Engineering